In [1]:
import torch, time, gc, csv
from transformers import BertConfig, BertForMaskedLM

def benchmark_config(batch_size, seq_len, num_layers, vocab_size=130,
                     hidden_size=768, num_heads=12,
                     n_warmup=3, n_steps=10, amp=True):
    """Returns (peak_mem_gb, step_ms) or (None, None) on OOM."""
    model = optimizer = scaler = input_ids = labels = None
    try:
        config = BertConfig(
            vocab_size=vocab_size,
            hidden_size=hidden_size,
            num_hidden_layers=num_layers,
            num_attention_heads=num_heads,
            intermediate_size=hidden_size * 4,
            max_position_embeddings=max(seq_len, 512),
            type_vocab_size=1,
        )
        model = BertForMaskedLM(config).cuda()
        optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4)
        scaler = torch.cuda.amp.GradScaler(enabled=amp)

        input_ids = torch.randint(0, vocab_size, (batch_size, seq_len), device='cuda')
        labels    = torch.randint(0, vocab_size, (batch_size, seq_len), device='cuda')

        def step():
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', enabled=amp):
                out = model(input_ids=input_ids, labels=labels)
            scaler.scale(out.loss).backward()
            scaler.step(optimizer)
            scaler.update()

        for _ in range(n_warmup): step()

        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        for _ in range(n_steps): step()
        torch.cuda.synchronize()
        t1 = time.perf_counter()

        return torch.cuda.max_memory_allocated() / (1024**3), (t1 - t0) / n_steps * 1000

    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if 'out of memory' not in str(e).lower(): raise
        return None, None
    finally:
        del model, optimizer, scaler, input_ids, labels
        gc.collect(); torch.cuda.empty_cache()


total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {torch.cuda.get_device_name(0)}  ({total_gb:.1f} GB)\n")

# (batch, seq, layers) grid — edit freely
configs = [(bs, seq, layers)
           for layers in [6, 12]
           for seq    in [512, 1024, 2048]
           for bs     in [16, 32, 64, 128, 256]]
configs.sort(key=lambda c: (c[2], c[1], c[0]))  # so smart-skip works

print(f"{'Layers':>7} {'Seq':>5} {'Batch':>6} {'Peak GB':>9} {'ms/step':>9} {'tok/s':>10}")
print("-" * 55)

oom_at, results = {}, []
for bs, seq, layers in configs:
    key = (layers, seq)
    if key in oom_at and bs >= oom_at[key]:
        print(f"{layers:>7} {seq:>5} {bs:>6} {'skip':>9} {'-':>9} {'-':>10}")
        continue
    mem, step_ms = benchmark_config(bs, seq, layers)
    if mem is None:
        oom_at[key] = bs
        print(f"{layers:>7} {seq:>5} {bs:>6} {'OOM':>9} {'-':>9} {'-':>10}")
        results.append({'layers':layers,'seq':seq,'batch':bs,'mem_gb':'','step_ms':'','tok_per_s':''})
    else:
        tok_s = bs * seq * 1000 / step_ms
        print(f"{layers:>7} {seq:>5} {bs:>6} {mem:>9.2f} {step_ms:>9.1f} {tok_s:>10,.0f}")
        results.append({'layers':layers,'seq':seq,'batch':bs,
                        'mem_gb':round(mem,2),'step_ms':round(step_ms,1),'tok_per_s':round(tok_s)})

with open('methylbert_bench.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=['layers','seq','batch','mem_gb','step_ms','tok_per_s'])
    w.writeheader(); w.writerows(results)
print(f"\nSaved methylbert_bench.csv ({len(results)} rows)")

GPU: NVIDIA RTX A6000  (47.4 GB)

 Layers   Seq  Batch   Peak GB   ms/step      tok/s
-------------------------------------------------------


/tmp/ipykernel_4006087/1416579229.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp)


      6   512     16      2.23      50.9    160,964


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


      6   512     32      3.83      90.3    181,421
      6   512     64      7.06     166.5    196,761
      6   512    128     13.53     316.8    206,901
      6   512    256     26.46     622.4    210,575
      6  1024     16      3.84      96.4    169,885
      6  1024     32      7.07     178.5    183,559
      6  1024     64     13.53     339.9    192,808
      6  1024    128     26.46     666.6    196,625
      6  1024    256       OOM         -          -
      6  2048     16      7.08     200.2    163,685
      6  2048     32     13.54     384.8    170,314
      6  2048     64     26.47     758.8    172,728
      6  2048    128       OOM         -          -
      6  2048    256      skip         -          -
     12   512     16      4.27      99.3     82,499
     12   512     32      7.36     175.2     93,537
     12   512     64     13.55     323.5    101,301
     12   512    128     25.93     616.3    106,343
     12   512    256       OOM         -          -
     12  102

In [2]:
import torch, time, gc, statistics
from transformers import BertConfig, BertForMaskedLM

def benchmark_detailed(batch_size, seq_len, num_layers, grad_accum,
                       vocab_size=130, hidden_size=768, num_heads=12,
                       n_warmup_opt=2, n_measured_opt=5, amp=True):
    """Benchmark with proper gradient accumulation. Times each microbatch
    AND each full optimizer step (= grad_accum microbatches + optimizer.step)."""
    try:
        config = BertConfig(
            vocab_size=vocab_size, hidden_size=hidden_size,
            num_hidden_layers=num_layers, num_attention_heads=num_heads,
            intermediate_size=hidden_size * 4,
            max_position_embeddings=max(seq_len, 512),
            type_vocab_size=1,
        )
        model = BertForMaskedLM(config).cuda()
        total_params = sum(p.numel() for p in model.parameters())

        optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4)
        scaler = torch.cuda.amp.GradScaler(enabled=amp)

        input_ids = torch.randint(0, vocab_size, (batch_size, seq_len), device='cuda')
        labels    = torch.randint(0, vocab_size, (batch_size, seq_len), device='cuda')

        def micro_fwd_bwd():
            with torch.autocast(device_type='cuda', enabled=amp):
                out = model(input_ids=input_ids, labels=labels)
                loss = out.loss / grad_accum
            scaler.scale(loss).backward()

        # Warmup
        for _ in range(n_warmup_opt):
            optimizer.zero_grad(set_to_none=True)
            for _ in range(grad_accum): micro_fwd_bwd()
            scaler.step(optimizer); scaler.update()

        # Measure
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        micro_times, opt_times = [], []

        for _ in range(n_measured_opt):
            torch.cuda.synchronize()
            t_opt0 = time.perf_counter()
            optimizer.zero_grad(set_to_none=True)
            for _ in range(grad_accum):
                torch.cuda.synchronize()
                t0 = time.perf_counter()
                micro_fwd_bwd()
                torch.cuda.synchronize()
                micro_times.append((time.perf_counter() - t0) * 1000)
            scaler.step(optimizer); scaler.update()
            torch.cuda.synchronize()
            opt_times.append((time.perf_counter() - t_opt0) * 1000)

        peak_gb = torch.cuda.max_memory_allocated() / (1024**3)
        return {'ok': True, 'total_params': total_params, 'peak_gb': peak_gb,
                'micro_ms': micro_times, 'opt_ms': opt_times}

    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if 'out of memory' not in str(e).lower(): raise
        return {'ok': False, 'error': 'OOM'}
    finally:
        for v in ('model','optimizer','scaler','input_ids','labels'):
            if v in dir(): pass
        gc.collect(); torch.cuda.empty_cache()


def report(name, cfg, r, gpu_gb):
    print(f"\n{'='*72}\n{name}")
    print(f"  layers={cfg['num_layers']}  batch={cfg['batch_size']}  "
          f"seq={cfg['seq_len']}  grad_accum={cfg['grad_accum']}  "
          f"effective_batch={cfg['batch_size']*cfg['grad_accum']}")
    print('='*72)
    if not r['ok']:
        print(f"  >>> {r['error']} <<<"); return

    print(f"\nParameters: {r['total_params']:,} ({r['total_params']/1e6:.1f}M)")
    print(f"Peak VRAM:  {r['peak_gb']:.2f} / {gpu_gb:.1f} GB  "
          f"({100*r['peak_gb']/gpu_gb:.0f}% utilization)")

    m, o = r['micro_ms'], r['opt_ms']
    print(f"\nMicro-batch (1 fwd+bwd over batch={cfg['batch_size']}, "
          f"n={len(m)} samples):")
    print(f"  mean={statistics.mean(m):6.1f} ms   "
          f"stdev={statistics.stdev(m):5.1f}   "
          f"min={min(m):6.1f}   max={max(m):6.1f}")

    print(f"\nOptimizer step ({cfg['grad_accum']} microbatches + AdamW update, "
          f"n={len(o)}):")
    print(f"  mean={statistics.mean(o):6.0f} ms   "
          f"stdev={statistics.stdev(o):5.0f}   "
          f"min={min(o):6.0f}   max={max(o):6.0f}")

    eff_tok = cfg['batch_size'] * cfg['seq_len'] * cfg['grad_accum']
    print(f"\nThroughput:  {eff_tok / (statistics.mean(o)/1000):>12,.0f} tokens/s")
    print(f"ETA @120k optimizer steps (paper): "
          f"{120_000 * statistics.mean(o) / 1000 / 3600:.1f} h "
          f"({120_000 * statistics.mean(o) / 1000 / 86400:.1f} d)")


gpu_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {torch.cuda.get_device_name(0)}  ({gpu_gb:.1f} GB)")

configs = [
    ('seq=512  / batch=128 / accum=8',
        dict(num_layers=12, seq_len=512,  batch_size=128, grad_accum=8)),
    ('seq=1024 / batch=64  / accum=16',
        dict(num_layers=12, seq_len=1024, batch_size=64,  grad_accum=16)),
]
for name, cfg in configs:
    report(name, cfg, benchmark_detailed(**cfg), gpu_gb)

GPU: NVIDIA RTX A6000  (47.4 GB)


/tmp/ipykernel_4006087/2412780607.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp)



seq=512  / batch=128 / accum=8
  layers=12  batch=128  seq=512  grad_accum=8  effective_batch=1024

Parameters: 86,142,082 (86.1M)
Peak VRAM:  26.27 / 47.4 GB  (55% utilization)

Micro-batch (1 fwd+bwd over batch=128, n=40 samples):
  mean= 603.2 ms   stdev=  2.1   min= 598.9   max= 606.0

Optimizer step (8 microbatches + AdamW update, n=5):
  mean=  4840 ms   stdev=   16   min=  4813   max=  4853

Throughput:       108,325 tokens/s
ETA @120k optimizer steps (paper): 161.3 h (6.7 d)

seq=1024 / batch=64  / accum=16
  layers=12  batch=64  seq=1024  grad_accum=16  effective_batch=1024

Parameters: 86,535,298 (86.5M)
Peak VRAM:  26.27 / 47.4 GB  (55% utilization)

Micro-batch (1 fwd+bwd over batch=64, n=80 samples):
  mean= 649.5 ms   stdev=  1.9   min= 645.6   max= 654.3

Optimizer step (16 microbatches + AdamW update, n=5):
  mean= 10407 ms   stdev=   29   min= 10378   max= 10449

Throughput:       100,758 tokens/s
ETA @120k optimizer steps (paper): 346.9 h (14.5 d)


In [3]:
import torch, time, gc, statistics
from transformers import BertConfig, BertForMaskedLM

def benchmark_detailed(batch_size, seq_len, num_layers, grad_accum,
                       vocab_size=130, hidden_size=768, num_heads=12,
                       n_warmup_opt=2, n_measured_opt=5,
                       n_warmup_eval=3, n_measured_eval=20, amp=True):
    """Times training (fwd+bwd+opt with grad accumulation) AND eval
    (fwd-only, no_grad) for a single config."""
    try:
        config = BertConfig(
            vocab_size=vocab_size, hidden_size=hidden_size,
            num_hidden_layers=num_layers, num_attention_heads=num_heads,
            intermediate_size=hidden_size * 4,
            max_position_embeddings=max(seq_len, 512),
            type_vocab_size=1,
        )
        model = BertForMaskedLM(config).cuda()
        total_params = sum(p.numel() for p in model.parameters())

        optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4)
        scaler = torch.cuda.amp.GradScaler(enabled=amp)

        input_ids = torch.randint(0, vocab_size, (batch_size, seq_len), device='cuda')
        labels    = torch.randint(0, vocab_size, (batch_size, seq_len), device='cuda')

        # ===== Training =====
        def micro_fwd_bwd():
            with torch.autocast(device_type='cuda', enabled=amp):
                out = model(input_ids=input_ids, labels=labels)
                loss = out.loss / grad_accum
            scaler.scale(loss).backward()

        model.train()
        for _ in range(n_warmup_opt):
            optimizer.zero_grad(set_to_none=True)
            for _ in range(grad_accum): micro_fwd_bwd()
            scaler.step(optimizer); scaler.update()

        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        micro_times, opt_times = [], []
        for _ in range(n_measured_opt):
            torch.cuda.synchronize()
            t_opt0 = time.perf_counter()
            optimizer.zero_grad(set_to_none=True)
            for _ in range(grad_accum):
                torch.cuda.synchronize()
                t0 = time.perf_counter()
                micro_fwd_bwd()
                torch.cuda.synchronize()
                micro_times.append((time.perf_counter() - t0) * 1000)
            scaler.step(optimizer); scaler.update()
            torch.cuda.synchronize()
            opt_times.append((time.perf_counter() - t_opt0) * 1000)
        train_peak_gb = torch.cuda.max_memory_allocated() / (1024**3)

        # ===== Eval =====
        model.eval()
        optimizer.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

        def eval_step():
            with torch.no_grad(), torch.autocast(device_type='cuda', enabled=amp):
                model(input_ids=input_ids, labels=labels)

        for _ in range(n_warmup_eval): eval_step()
        torch.cuda.synchronize()
        eval_times = []
        for _ in range(n_measured_eval):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            eval_step()
            torch.cuda.synchronize()
            eval_times.append((time.perf_counter() - t0) * 1000)
        eval_peak_gb = torch.cuda.max_memory_allocated() / (1024**3)

        return {'ok': True, 'total_params': total_params,
                'train_peak_gb': train_peak_gb, 'eval_peak_gb': eval_peak_gb,
                'micro_ms': micro_times, 'opt_ms': opt_times, 'eval_ms': eval_times}

    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if 'out of memory' not in str(e).lower(): raise
        return {'ok': False, 'error': 'OOM'}
    finally:
        gc.collect(); torch.cuda.empty_cache()


def report(name, cfg, r, gpu_gb,
           total_steps=120_000, eval_freq=1000, test_set_reads=16_000_000):
    print(f"\n{'='*72}\n{name}")
    print(f"  layers={cfg['num_layers']}  batch={cfg['batch_size']}  "
          f"seq={cfg['seq_len']}  grad_accum={cfg['grad_accum']}  "
          f"effective_batch={cfg['batch_size']*cfg['grad_accum']}")
    print('='*72)
    if not r['ok']:
        print(f"  >>> {r['error']} <<<"); return

    print(f"\nParameters: {r['total_params']:,} ({r['total_params']/1e6:.1f}M)")
    print(f"Peak VRAM train: {r['train_peak_gb']:5.2f} / {gpu_gb:.1f} GB "
          f"({100*r['train_peak_gb']/gpu_gb:.0f}%)")
    print(f"Peak VRAM eval:  {r['eval_peak_gb']:5.2f} / {gpu_gb:.1f} GB "
          f"({100*r['eval_peak_gb']/gpu_gb:.0f}%)")

    m, o, e = r['micro_ms'], r['opt_ms'], r['eval_ms']
    print(f"\nTrain microbatch (batch={cfg['batch_size']}, n={len(m)}):")
    print(f"  mean={statistics.mean(m):6.1f} ms  stdev={statistics.stdev(m):5.1f}  "
          f"min={min(m):6.1f}  max={max(m):6.1f}")
    print(f"\nOptimizer step ({cfg['grad_accum']} microbatches + AdamW, n={len(o)}):")
    print(f"  mean={statistics.mean(o):6.0f} ms  stdev={statistics.stdev(o):5.0f}  "
          f"min={min(o):6.0f}  max={max(o):6.0f}")
    print(f"\nEval batch (fwd-only, batch={cfg['batch_size']}, n={len(e)}):")
    print(f"  mean={statistics.mean(e):6.1f} ms  stdev={statistics.stdev(e):5.1f}  "
          f"min={min(e):6.1f}  max={max(e):6.1f}  "
          f"({statistics.mean(e)/statistics.mean(m)*100:.0f}% of train microbatch)")

    train_step_s = statistics.mean(o) / 1000
    eval_batch_s = statistics.mean(e) / 1000
    n_evals = total_steps // eval_freq
    n_full = test_set_reads // cfg['batch_size']

    train_d = total_steps * train_step_s / 86400
    full_d  = n_evals * n_full * eval_batch_s / 86400

    print(f"\n--- ETAs ({total_steps:,} steps, eval every {eval_freq:,}, "
          f"test set = {test_set_reads:,} reads → {n_evals} evals) ---")
    print(f"  Training only:           {train_d:6.2f} d ({train_d*24:5.1f} h)")
    print(f"  Eval, full test set:     {full_d:6.2f} d ({full_d*24:5.1f} h)  "
          f"[{n_full:,} batches/eval]")
    for cap in [500, 1000, 2000, 5000]:
        d = n_evals * cap * eval_batch_s / 86400
        print(f"  Eval, capped @ {cap:>5}:   {d:6.2f} d ({d*24:5.1f} h)")
    print(f"\n  Total w/ full eval:      {train_d + full_d:.2f} d")
    print(f"  Total w/ cap 1000:       {train_d + n_evals*1000*eval_batch_s/86400:.2f} d")


gpu_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {torch.cuda.get_device_name(0)}  ({gpu_gb:.1f} GB)")

configs = [
    ('seq=512  / batch=128 / accum=8',
        dict(num_layers=12, seq_len=512,  batch_size=128, grad_accum=8)),
    ('seq=1024 / batch=64  / accum=16',
        dict(num_layers=12, seq_len=1024, batch_size=64,  grad_accum=16)),
]
for name, cfg in configs:
    report(name, cfg, benchmark_detailed(**cfg), gpu_gb)

GPU: NVIDIA RTX A6000  (47.4 GB)


/tmp/ipykernel_4006087/3249670102.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp)



seq=512  / batch=128 / accum=8
  layers=12  batch=128  seq=512  grad_accum=8  effective_batch=1024

Parameters: 86,142,082 (86.1M)
Peak VRAM train: 26.27 / 47.4 GB (55%)
Peak VRAM eval:   2.57 / 47.4 GB (5%)

Train microbatch (batch=128, n=40):
  mean= 601.9 ms  stdev=  2.1  min= 596.9  max= 605.0

Optimizer step (8 microbatches + AdamW, n=5):
  mean=  4829 ms  stdev=   16  min=  4802  max=  4842

Eval batch (fwd-only, batch=128, n=20):
  mean= 187.8 ms  stdev=  0.5  min= 187.3  max= 188.8  (31% of train microbatch)

--- ETAs (120,000 steps, eval every 1,000, test set = 16,000,000 reads → 120 evals) ---
  Training only:             6.71 d (161.0 h)
  Eval, full test set:      32.61 d (782.7 h)  [125,000 batches/eval]
  Eval, capped @   500:     0.13 d (  3.1 h)
  Eval, capped @  1000:     0.26 d (  6.3 h)
  Eval, capped @  2000:     0.52 d ( 12.5 h)
  Eval, capped @  5000:     1.30 d ( 31.3 h)

  Total w/ full eval:      39.32 d
  Total w/ cap 1000:       6.97 d

seq=1024 / batch=64  